# 01 · Carga y exploración

**Tiempo estimado:** 30 min.

**Objetivos.** Cargar las series del curso, inspeccionar su estructura temporal (índice, frecuencia, gaps, rango), y producir las visualizaciones de referencia que volveremos a ver en sesiones posteriores.

**Datasets** (cargados por `cst.datos`):

- `caudal`: caudal diario del Genil en el cluster SAIH `A20_GENIL_TOCON` (≈2018-presente). Para la serie histórica ROEA 5020 (1995-2020) pasar `source="CEDEX"`.
- `lluvia`: precipitación diaria del pluviómetro SAIH `A20_202` (mismo cluster A20).
- `piezo`: cota piezométrica de `PZ0267014` (CHD).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from cst import datos as ud

plt.rcParams.update({"figure.figsize": (10, 3.4), "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
caudal = ud.cargar_caudal_genil()
lluvia = ud.cargar_lluvia_genil()
piezo = ud.cargar_piezometria()

for nombre, s in [("caudal", caudal), ("lluvia", lluvia), ("piezo", piezo)]:
    print(f"\n=== {nombre} ===")
    print(ud.resumen(s))

## 1 · Estructura del índice temporal

Lo primero que debe revisar cualquier análisis de series:

- ¿Cuál es la frecuencia **nominal** y la **real**? (¿faltan días?)
- ¿Hay duplicados de timestamp?
- ¿Hay valores fuera de rango físico?


In [ ]:
print("Frecuencia inferida (caudal):", pd.infer_freq(caudal.index[:50]))
print("Duplicados de fecha (caudal):", caudal.index.duplicated().sum())
print("Rango temporal (caudal):    ", caudal.index.min().date(), "→", caudal.index.max().date())

# Diferencias entre timestamps consecutivos
diffs = caudal.index.to_series().diff().dropna().value_counts().head(5)
print("\nDistribución de gaps entre observaciones:")
print(diffs)

In [ ]:
# La piezometría es irregular: ver distribución de intervalos
intervalos = piezo.index.to_series().diff().dt.days.dropna()
print(
    f"Piezometría: {len(piezo)} mediciones entre {piezo.index.min().year} y {piezo.index.max().year}"
)
print(intervalos.describe().astype(int))

fig, ax = plt.subplots(figsize=(8, 3))
intervalos.clip(upper=400).hist(bins=40, ax=ax, color="#0d9488", alpha=0.8)
ax.set_xlabel("Intervalo entre mediciones (días, recortado a 400)")
ax.set_ylabel("Frecuencia")
ax.set_title("Cadencia real de medición de PZ0267014")
plt.tight_layout()

## 2 · Plots generales

Una primera mirada a la serie completa: nivel medio, varianza, eventos claros.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=False)

axes[0].plot(caudal, color="#1f6f8b", lw=0.4)
axes[0].set_ylabel("Q (m³/s)")
axes[0].set_title("Caudal diario · SAIH A20_GENIL_TOCON (2018–presente)")

axes[1].plot(lluvia, color="#2563eb", lw=0.8)
axes[1].set_ylabel("P diaria (mm)")
axes[1].set_title("Lluvia diaria · SAIH A20_202")

axes[2].plot(piezo, color="#0d9488", marker="o", ms=3, lw=0.6)
axes[2].set_ylabel("Cota (m s.n.m.)")
axes[2].set_title("Cota piezométrica · PZ0267014")

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()

## 3 · Zoom: crecida del Genil — febrero 2026

El máximo de la serie SAIH ocurre en **febrero de 2026** (>300 m³/s). Es el tipo de evento que querremos predecir.

In [ ]:
ventana = caudal.loc["2026-02-01":"2026-03-15"]

fig, ax = plt.subplots()
ax.plot(ventana.index, ventana.values, color="#c2410c", lw=1.5, marker="o", ms=4)
ax.axhline(caudal.dropna().quantile(0.95), color="grey", ls="--", lw=0.8, label="P95 global")
ax.set_xlabel("Fecha")
ax.set_ylabel("Caudal (m³/s)")
ax.set_title("Crecida del Genil — febrero 2026")
ax.legend()
plt.tight_layout()

print("Pico:", ventana.idxmax().date(), "→", ventana.max(), "m³/s")
print("Días por encima del P99 histórico:", (ventana > caudal.dropna().quantile(0.99)).sum())

## 4 · Estadísticos descriptivos por régimen

Caudal y piezometría tienen estacionalidad anual marcada. Conviene mirar las distribuciones por mes/estación, no solo el agregado.


In [ ]:
q = caudal.loc["2020":"2025"].dropna()
df_q = q.to_frame("Q")
df_q["mes"] = df_q.index.month

fig, ax = plt.subplots(figsize=(9, 3.6))
df_q.boxplot(column="Q", by="mes", ax=ax, showfliers=False)
ax.set_yscale("log")
ax.set_xlabel("Mes")
ax.set_ylabel("Q (m³/s, escala log)")
ax.set_title("Distribución mensual del caudal · Pinos-Genil")
plt.suptitle("")
plt.tight_layout()

Régimen pluvio-nival típico del Genil: caudales medios altos en invierno–primavera (deshielo de Sierra Nevada en mayo–junio históricamente) y estiaje pronunciado en verano.


## 5 · Año hidrológico vs año civil

En España el año hidrológico va de **octubre a septiembre**. Si vamos a hacer agregados anuales, conviene hacerlos así.


In [ ]:
# pandas: 'YE-SEP' termina el año en septiembre (es decir, año hidrológico oct→sep etiquetado con el año del final)
q_anual_hidro = caudal.resample("YE-SEP").mean()

fig, ax = plt.subplots()
ax.plot(q_anual_hidro.index.year, q_anual_hidro.values, marker="o", color="#1f6f8b")
ax.set_xlabel("Año hidrológico (terminado en septiembre)")
ax.set_ylabel("Q medio anual (m³/s)")
ax.set_title("Caudal medio anual hidrológico · Cluster SAIH A20")
plt.tight_layout()

## 6 · Ejercicios del bloque

1. **Año más seco vs más húmedo.** Identifica el año hidrológico con caudal medio más bajo y el más alto. Calcula el ratio.
2. **Cobertura por estación.** Para `PZ0267014`, ¿cuántas mediciones hay por década? ¿Cambia la cadencia?
3. **Cuantiles condicionados.** Calcula el percentil 95 del caudal **por mes**. ¿En qué mes hace falta más caudal para ser "alto"?
4. **Reto.** El catálogo `ud.cargar_anuario_estaciones()` tiene la columna `cod_saih`. Empareja todas las estaciones ROEA del Guadalquivir con su equivalente SAIH y dibuja en mapa (con `xetrs89/yetrs89`) las que tienen ambos códigos.
5. **Crecida 1985 vs 2026 (ROEA + SAIH).** Carga la serie histórica ROEA con `ud.cargar_caudal_genil(source="CEDEX")` y haz el zoom equivalente al de febrero 2026 sobre el evento de febrero 1985. Compara la magnitud relativa de ambos picos respecto al P95 de su periodo de referencia (1995-2020 vs 2018-presente). ¿Cuál es relativamente más extremo?
6. **Tres fuentes de lluvia diaria.** Carga la lluvia SAIH `A20_202` (`ud.cargar_lluvia_genil()`) y la lluvia ERA5 sobre Pinos-Genil (`ud.cargar_lluvia_genil_diaria(fecha_inicio="2018-01-01", fecha_fin="2024-12-31")`). Solapa el periodo y compara: (a) acumulado total, (b) número de días con lluvia > 1 mm, (c) correlación día-a-día. ¿Coinciden? ¿Qué fuente mide más? Discute por qué (resolución espacial ERA5 25 km vs pluviómetro puntual).

Las soluciones propuestas están en `ejercicios.md`.